In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!which ollama || echo 'NOT ON PATH'

In [ ]:
import subprocess, time, urllib.request

subprocess.Popen(['ollama', 'serve'])
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        print('Ollama server is up')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama server did not come up')

In [ ]:
!git clone https://github.com/orceal-lab/cot-failure-modes.git
%cd cot-failure-modes
!pip install -q -r requirements.txt

In [ ]:
!ollama pull qwen3:4b

In [ ]:
# Rerun of just qwen3:4b on arithmetic_singles_hard.json -- the first
# ceiling-effect run (cot-ceiling-run) got mistral, qwen2.5-coder, and
# llama3.1:8b cleanly, but qwen3:4b hit a 600s read-timeout at problem
# 118/120 (12-fact level) and crashed with no saved output. Bumped the
# Ollama client timeout to 1200s (models/ollama_client.py) since qwen3's
# thinking-mode traces can run long, and rerunning just this one model.
!python run_experiment.py --problems problems/arithmetic_singles_hard.json --model \
  "ollama:qwen3:4b"

In [ ]:
!python analyze.py results/raw_*_arithmetic_singles_hard_*.json --results-dir results/singles_hard_qwen3
!python compare_models.py results/singles_hard_qwen3/analysis_*.csv --results-dir results/singles_hard_qwen3

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', 'results')
print('Zipped results to /kaggle/working/results.zip')